# Análise Exploratória — Silver: ecommerce_clientes

Exploração da tabela Silver de clientes (`squad1/silver/ecommerce_clientes` no ADLS): volume e versões por cliente, completude das colunas, provedores de e-mail (incluindo descartáveis), data quality das 10 regras `regra_NN_falhou` e duplicações de e-mail/nome. Complementa a EDA de endereços do notebook 08 e alimenta o modelo de anomalias do notebook 10.

### 1. Configuração e leitura da Silver
Credenciais do ADLS carregadas do `.env` via API do workspace (no serverless, `load_dotenv()` não acessa arquivos ocultos) e leitura da Delta `squad1/silver/ecommerce_clientes`. Como a Silver é append por micro-lote, mantém-se a última versão de cada `id_cliente` (deduplicação por `bronze_source_file`).

In [0]:
# 1. Credenciais ADLS (.env via API do workspace) e leitura da Silver
import os
from databricks.sdk import WorkspaceClient
from pyspark.sql import Window
from pyspark.sql import functions as F

ENV_PATH = "/Users/luizpopoff@gmail.com/estagio-empregadados-turma-2/.env"

# No serverless o FUSE não expõe arquivos ocultos: baixa o .env pela API do workspace
_env_txt = WorkspaceClient().workspace.download(ENV_PATH).read().decode()
for _linha in _env_txt.splitlines():
    _linha = _linha.strip()
    if _linha and not _linha.startswith("#") and "=" in _linha:
        _k, _v = _linha.split("=", 1)
        os.environ.setdefault(_k.strip(), _v.strip())

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.environ["ADLS_CLIENT_ID"],
    "fs.azure.account.oauth2.client.secret": os.environ["ADLS_CLIENT_SECRET"],
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.environ['ADLS_TENANT_ID']}/oauth2/token",
}
clientes_path = "abfss://squad1@internshipdatalake.dfs.core.windows.net/silver/ecommerce_clientes"

df_silver_raw = spark.read.format("delta").options(**adls_opts).load(clientes_path)

# Última versão de cada cliente (Silver é append por micro-lote)
_w = Window.partitionBy("id_cliente").orderBy(F.col("bronze_source_file").desc())
df_silver = (
    df_silver_raw
    .withColumn("_versao", F.row_number().over(_w))
    .filter(F.col("_versao") == 1)
    .drop("_versao")
)

print(f"Linhas na Silver (todas as versões): {df_silver_raw.count()}")
print(f"Clientes distintos (última versão):  {df_silver.count()}")
df_silver.printSchema()

### 2. Visão geral e completude
Volume, chaves, quantas versões cada cliente acumula na Silver e nulos por coluna de negócio (ignora colunas técnicas `regra_*`, `bronze_*`, `silver_*` e as particionadoras `ano/mes/dia`).

In [0]:
# 2. Visão geral: volume, chaves, versões e completude
COLS_NEGOCIO = [
    c for c in df_silver.columns
    if not c.startswith(("regra_", "bronze_", "silver_")) and c not in ("ano", "mes", "dia")
]

print(f"Total de clientes (dedup):     {df_silver.count()}")
print(f"Clientes distintos (id):        {df_silver.select('id_cliente').distinct().count()}")
print(f"UUIDs distintos:               {df_silver.select('uuid_cliente').distinct().count()}")
print(f"E-mails distintos:             {df_silver.select('email').distinct().count()}")
print(f"Versões acumuladas na Silver:  {df_silver_raw.count()}")

# Quantas versões cada cliente acumula na Silver bruta
versoes = df_silver_raw.groupBy("id_cliente").count().withColumnRenamed("count", "qtd_versoes")
display(versoes.groupBy("qtd_versoes").count().orderBy("qtd_versoes"))

# Nulos por coluna de negócio (última versão de cada cliente)
nulls_df = df_silver.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in COLS_NEGOCIO]
)
display(nulls_df)

### 3. Distribuições
Provedores de e-mail (atenção aos descartáveis), status da data de cadastro e evolução dos cadastros no tempo.

In [0]:
# 3. Distribuições: provedor de e-mail, status do cadastro e cadastro temporal
display(df_silver.groupBy("provedor_email").count().orderBy(F.desc("count")))

display(df_silver.groupBy("status_dt_cadastro").count())

# Cadastros por ano/mês
display(
    df_silver
    .groupBy(F.year("dt_cadastro").alias("ano"), F.month("dt_cadastro").alias("mes"))
    .count()
    .orderBy("ano", "mes")
)

# Estatísticas das datas
display(
    df_silver.select("dt_cadastro", "dt_ultima_atualizacao").summary(
        "count", "min", "25%", "50%", "75%", "max"
    )
)

### 4. Data Quality
Percentual de clientes com falha (`True`) em cada uma das 10 regras `regra_NN_falhou` gravadas pela Silver de clientes, do pior para o melhor, e o rótulo fraco consolidado (falhou em ao menos uma regra).

In [0]:
# 4. Taxa de falha por regra de DQ (regra_NN_falhou: True = falhou) e rótulo fraco
REGRAS = [c for c in df_silver.columns if c.startswith("regra_")]

if not REGRAS:
    print("Nenhuma coluna regra_NN_falhou encontrada na Silver.")
else:
    _falhas = df_silver.agg(
        *[F.sum(F.coalesce(F.col(c), F.lit(False)).cast("long")).alias(c) for c in REGRAS]
    ).collect()[0]
    total = df_silver.count()
    falhas_df = spark.createDataFrame(
        [(c, int(_falhas[c])) for c in REGRAS], ["regra", "qtd_falhas"]
    ).withColumn("pct_falha", F.round(F.col("qtd_falhas") / total * 100, 2))
    display(falhas_df.orderBy(F.desc("qtd_falhas")))

    # Rótulo fraco consolidado (1 = falhou em ao menos uma regra)
    _alguma = F.greatest(
        *[F.coalesce(F.col(c), F.lit(False)).cast("int") for c in REGRAS]
    )
    df_silver = df_silver.withColumn("rotulo_fraco", _alguma)
    n_rot = df_silver.filter(F.col("rotulo_fraco") == 1).count()
    print(f"\nClientes com ao menos uma falha (rótulo fraco): {n_rot} ({n_rot / total * 100:.2f}%)")

### 5. Duplicações e sinais de risco
E-mails e nomes completos compartilhados por mais de um cliente, e clientes com provedores descartáveis (`tempmail.xyz`, `mailfake.io`, `xmail99.net`) — sinais usados pelo modelo de anomalias do notebook 10.

In [0]:
# 5. Duplicações e sinais de risco
# E-mails usados por mais de um cliente distinto
dup_email = (
    df_silver.groupBy("email")
    .agg(F.countDistinct("id_cliente").alias("qtd_clientes"))
    .filter(F.col("qtd_clientes") > 1)
)
print(f"E-mails compartilhados por mais de um cliente: {dup_email.count()}")
display(dup_email.orderBy(F.desc("qtd_clientes")))

# Nome completo repetido entre clientes distintos
dup_nome = (
    df_silver
    .groupBy(F.concat_ws(" ", "nome", "sobrenome").alias("nome_completo"))
    .agg(F.countDistinct("id_cliente").alias("qtd_clientes"))
    .filter(F.col("qtd_clientes") > 1)
)
print(f"Nomes completos compartilhados por mais de um cliente: {dup_nome.count()}")

# Clientes com provedor de e-mail descartável
DESCARTAVEIS = ["tempmail.xyz", "mailfake.io", "xmail99.net"]
susp = df_silver.filter(F.lower(F.col("provedor_email")).isin(DESCARTAVEIS))
print(f"\nClientes com provedor descartável: {susp.count()}")
display(susp.select("id_cliente", "nome", "sobrenome", "email", "provedor_email"))

### Síntese (execução de 2026-11-11)

**Volume e versões:** 1.182 versões acumuladas para 1.135 clientes distintos — 47 clientes têm 2 versões na Silver (dedup por `bronze_source_file` mantém a última). Chaves íntegras: 1.135 ids, 1.135 UUIDs, zero nulos em todas as colunas de negócio.

**E-mail:** 9 provedores; `gmail.com` (520), `hotmail.com` (261) e `outlook.com` (195) dominam. **75 clientes (6,6%) usam provedores descartáveis** (tempmail.xyz 27, mailfake.io 25, xmail99.net 23) — os mesmos que o modelo de anomalias do notebook 10 detecta. Apenas 3 e-mails são compartilhados por 2 clientes cada (1.132 e-mails distintos), e nenhum nome completo se repete.

**Cadastro:** `status_dt_cadastro` é 100% VALIDO; cadastros distribuem-se por 37 meses (out/2023 a out/2026), sem lacunas mensais, com pico em out/2026 (51).

**Data Quality:** rótulo fraco em 348 clientes (30,66%). Principais falhas: `regra_08` (214; 18,85%), `regra_09` (75; 6,61%), `regra_02` (53; 4,67%), `regra_01` e `regra_10` (47 cada; 4,14%), `regra_04` (30; 2,64%). Regras 03, 05, 06 e 07 com zero falhas.

**Nota:** o rótulo fraco de clientes (30,66%) é bem menor que o de endereços (77%) — somar as flags `regra_NN_falhou` ao rótulo do notebook 10 tende a converter mais alertas de domínio em VP do que a criar novos FN.